In [1]:
import rectanglepy as rectangle
import pandas as pd
from anndata import AnnData
import h5py
import spatialdata_io
import helper
import biomart

In [2]:


counts = pd.read_csv('../data/human_m1/matrix.csv', index_col=0)
counts = counts.astype(int)

metadata = pd.read_csv('../data/human_m1/metadata.csv', index_col=0)


In [3]:
annotations = metadata['cell_type_alias_label']
# remove anything after the second space(word)
annotations = annotations.str.split(' ', n=2).str[:2].str.join(' ')
# remove anything afert first minus
annotations = annotations.str.split('-', n=1).str[0]

In [4]:
annotations.value_counts()

cell_type_alias_label
Exc L2      20187
Exc L3      14958
Inh L1       9692
Exc L5       9434
Inh L5       4984
Inh L2       4832
Inh L3       4400
Exc L6       3957
Oligo L2     2401
Astro L1      568
Oligo L3      535
OPC L1        283
Micro L1      108
Inh L6         84
Endo L2        64
VLMC L1        40
Oligo L5        6
Name: count, dtype: int64

In [5]:
adata = AnnData(counts, obs=annotations.to_frame(name='cell_type'))

In [6]:
f = spatialdata_io.visium(path='../data', dataset_id='151673',scalefactors_file='scalefactors_json.json', tissue_positions_file='tissue_positions.csv', counts_file='151673_filtered_feature_bc_matrix.h5')


INFO     reading ../data/151673_filtered_feature_bc_matrix.h5                                                      


/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


In [17]:
data_table = f['table']
bulks = data_table.to_df()
# Convert bulks from counts to CPM
bulks_cpm = bulks.div(bulks.sum(axis=1), axis=0) * 1e6

In [ ]:
estimations, signature_result = rectangle.rectangle(adata, bulks_cpm)

In [20]:
estimations.to_csv('../output/spatial_estimations.csv')

In [21]:
import pickle
with open('../output/signature_result_spatial.pkl', 'wb') as f:
    pickle.dump(signature_result, f)

In [30]:
from spatialdata.models import TableModel

estimations = pd.read_csv('../output/spatial_estimations.csv', index_col=0)

# Create an AnnData object from the estimations

table_data = AnnData(estimations)
adata_for_sdata = TableModel.parse(table_data)

adata_for_sdata.uns["spatialdata_attrs"] = {
    "region": "spots",
    "region_key": "region",  # column in adata.obs that will link a given obs to the elements it annotates
    "instance_key": "spot_id",  # column that matches a given obs in the table to a given circle
}

adata_for_sdata.obs["region"] = pd.Categorical(["spots"] * len(adata_for_sdata))
adata_for_sdata.obs["spot_id"] = f.tables.data['table'].obs["spot_id"]

f.tables["rectangle_results"] = adata_for_sdata





/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/spatialdata/_core/_elements.py:125: UserWarning: Key `rectangle_results` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
